In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Imports
import os
import glob
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
DATA_DIR = "/content/drive/MyDrive/Data"
csv_path = os.path.join(DATA_DIR, "Data_Entry_2017.csv")
image_dirs = [os.path.join(DATA_DIR, "images_001/images"),os.path.join(DATA_DIR, "images_002/images"),os.path.join(DATA_DIR, "images_003/images"),os.path.join(DATA_DIR, "images_004/images")]

In [ ]:
df = pd.read_csv(csv_path)
print("Original CSV shape:", df.shape)
print(df.head())

Original CSV shape: (112120, 12)
        Image Index          Finding Labels  Follow-up #  Patient ID  \
0  00000001_000.png            Cardiomegaly            0           1   
1  00000001_001.png  Cardiomegaly|Emphysema            1           1   
2  00000001_002.png   Cardiomegaly|Effusion            2           1   
3  00000002_000.png              No Finding            0           2   
4  00000003_000.png                  Hernia            0           3   

   Patient Age Patient Gender View Position  OriginalImage[Width  Height]  \
0           58              M            PA                 2682     2749   
1           58              M            PA                 2894     2729   
2           58              M            PA                 2500     2048   
3           81              M            PA                 2500     2048   
4           81              F            PA                 2582     2991   

   OriginalImagePixelSpacing[x     y]  Unnamed: 11  
0                 

In [ ]:
image_paths = {}
for folder in image_dirs:
    for img_path in glob.glob(os.path.join(folder, "**", "*.png"), recursive=True):
        fname = os.path.basename(img_path)
        image_paths[fname] = img_path
print("Total images found:", len(image_paths))

Total images found: 14999


In [ ]:
df = df[df["Image Index"].isin(image_paths.keys())].copy()
df["image_path"] = df["Image Index"].map(image_paths)
print("Rows after filtering available images:", len(df))

Rows after filtering available images: 14999


In [ ]:
df["labels"] = df["Finding Labels"].astype(str).apply(lambda x: x.split("|"))
print(df[["Finding Labels", "labels"]].head())

           Finding Labels                     labels
0            Cardiomegaly             [Cardiomegaly]
1  Cardiomegaly|Emphysema  [Cardiomegaly, Emphysema]
2   Cardiomegaly|Effusion   [Cardiomegaly, Effusion]
3              No Finding               [No Finding]
4                  Hernia                   [Hernia]


In [ ]:
df = df[df["Finding Labels"] != "No Finding"].copy()
print("Rows after removing 'No Finding':", len(df))

Rows after removing 'No Finding': 6246


In [ ]:
SAMPLE_SIZE = None
if SAMPLE_SIZE is not None:
    n = min(SAMPLE_SIZE, len(df))
    df = df.sample(n, random_state=42).reset_index(drop=True)
else:
    df = df.reset_index(drop=True)
print("Final dataset size:", len(df))

Final dataset size: 6246


In [ ]:
# Encode labels
mlb = MultiLabelBinarizer()
label_matrix = mlb.fit_transform(df["labels"])
print("Classes:")
print(list(mlb.classes_))
print("Label matrix shape:", label_matrix.shape)

Classes:
['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']
Label matrix shape: (6246, 14)


In [ ]:
# Add encoded labels
df["encoded_labels"] = list(label_matrix)

In [ ]:
# Remove corrupt images
valid_indices = []
for i, row in df.iterrows():
    try:
        with Image.open(row["image_path"]) as img:
            img.verify()
        valid_indices.append(i)
    except Exception:
        pass
df = df.loc[valid_indices].reset_index(drop=True)
print("Rows after removing corrupt images:", len(df))

Rows after removing corrupt images: 6246


In [ ]:
# Train / Val / Test split
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (4996, 15)
Val: (625, 15)
Test: (625, 15)


In [ ]:
processed_dir = os.path.join(DATA_DIR, "processed")
os.makedirs(processed_dir, exist_ok=True)

train_df.to_csv(os.path.join(processed_dir, "train.csv"), index=False)
val_df.to_csv(os.path.join(processed_dir, "val.csv"), index=False)
test_df.to_csv(os.path.join(processed_dir, "test.csv"), index=False)

classes_df = pd.DataFrame({"label_name": mlb.classes_})
classes_df.to_csv(os.path.join(processed_dir, "label_classes.csv"), index=False)

print("Saved files to:", processed_dir)
print(os.listdir(processed_dir))

Saved files to: /content/drive/MyDrive/Data/processed
['label_classes.csv', 'test.csv', 'val.csv', 'train.csv']


In this step, we prepared the NIH Chest X-ray dataset for training a multi-label classification model. We first loaded the Data_Entry_2017.csv file and extracted the corresponding X-ray images from the downloaded folders (images_001/images, images_002/images). Since the images were stored in nested directories, we used recursive search to map each image filename to its full file path.

We then filtered the dataset to include only those entries for which the images were available locally. The Finding Labels column was processed to convert label strings (e.g., "Effusion|Mass") into a list format. Rows with "No Finding" were removed to focus only on abnormal cases, which are more relevant for our task.

Next, we applied multi-label encoding using MultiLabelBinarizer to convert the labels into a binary vector format suitable for training. We also performed a basic data cleaning step to remove any corrupt images.

Finally, the dataset was split into training, validation, and test sets in an 80-10-10 ratio. The processed data, along with label mappings, was saved to disk for use in the model training stage.

Analyze Head of Table

In [ ]:
train_df.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11,image_path,labels,encoded_labels
4586,00002774_000.png,Infiltration,0,2774,30,M,PA,2048,2500,0.171,0.171,NaN,/content/drive/MyDrive/Data/images_002/images/...,[Infiltration],"[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]"
2251,00001355_002.png,Atelectasis,2,1355,43,F,AP,3056,2544,0.139,0.139,NaN,/content/drive/MyDrive/Data/images_002/images/...,[Atelectasis],"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4653,00002842_004.png,Effusion,4,2842,58,F,AP,2500,2048,0.171,0.171,NaN,/content/drive/MyDrive/Data/images_002/images/...,[Effusion],"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
605,00000250_015.png,Mass,15,250,52,M,PA,2992,2885,0.143,0.143,NaN,/content/drive/MyDrive/Data/images_001/images/...,[Mass],"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]"
1194,00000627_028.png,Atelectasis,28,627,59,M,AP,2500,2048,0.168,0.168,NaN,/content/drive/MyDrive/Data/images_001/images/...,[Atelectasis],"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


Training Class Distribution

In [ ]:
train_df['Finding Labels'].value_counts()

,count
Finding Labels,
Infiltration,911
Atelectasis,444
Effusion,376
Pneumothorax,296
Nodule,255
...,...
Cardiomegaly|Consolidation|Edema|Effusion,1
Consolidation|Mass|Nodule,1
Atelectasis|Cardiomegaly|Consolidation,1


Validation Class Distibution

In [ ]:
val_df['Finding Labels'].value_counts()

,count
Finding Labels,
Infiltration,118
Atelectasis,56
Effusion,44
Nodule,36
Pneumothorax,29
...,...
Effusion|Pneumonia|Pneumothorax,1
Edema|Nodule,1
Emphysema|Infiltration,1


Test Class Distribution

In [ ]:
test_df['Finding Labels'].value_counts()

,count
Finding Labels,
Infiltration,107
Atelectasis,57
Effusion,45
Pneumothorax,30
Nodule,30
...,...
Fibrosis|Infiltration|Nodule,1
Atelectasis|Cardiomegaly|Effusion,1
Emphysema|Infiltration|Pneumothorax,1


Combined Distribution (Percentage)

In [ ]:
combined_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
combined_df['Finding Labels'].value_counts(normalize=True) * 100

,proportion
Finding Labels,
Infiltration,18.187640
Atelectasis,8.917707
Effusion,7.444765
Pneumothorax,5.683638
Nodule,5.139289
...,...
Atelectasis|Consolidation|Effusion|Mass|Nodule,0.016010
Cardiomegaly|Edema|Infiltration,0.016010
Atelectasis|Cardiomegaly|Infiltration|Pneumothorax,0.016010


View Position Distribution

In [ ]:
train_df['View Position'].value_counts()

,count
View Position,
PA,2904
AP,2092


Age Distribution

In [ ]:
train_df['Patient Age'].describe()

,Patient Age
count,4996.000000
mean,50.412330
std,16.193528
min,6.000000
25%,40.000000
50%,52.000000
75%,61.000000
max,94.000000


Gender Distribution

In [ ]:
train_df['Patient Gender'].value_counts()

,count
Patient Gender,
M,2531
F,2465
